# Mobile Money Agent Network: Audit and Clean-Up

The regional manager wants to know which agents and counties handle the most deposits and withdrawals. The extract from the legacy system is unreliable, so this notebook **audits** the data, **cleans** it with a reusable function (`cleaning_pipeline.py`), **validates** the merge with the agent register and then **summarises** it. The data governance team can read the cleaning log and the Data Quality Report at the end.

In [1]:
# -- Setup --
import os
import subprocess
import sys

import numpy as np
import pandas as pd

from cleaning_pipeline import clean_agent_data

# show wide tables in full
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 70)

# make the data if it is not here yet
if not os.path.exists("agent_transactions.csv"):
    subprocess.run([sys.executable, "generate_data.py"], check=True)

os.makedirs("outputs", exist_ok=True)

## 1. Load the raw data and take a first look

I load the two files exactly as they are. I also load a second copy of the transactions with every value kept as text and blanks left alone (`keep_default_na=False`). By default pandas turns the text `N/A` into a missing value, so the text copy is the only way to tell a blank from an `N/A`.

In [2]:
# -- Load the raw files --
txns_raw = pd.read_csv("agent_transactions.csv")
agents = pd.read_csv("agents.csv")

# the same file with everything kept as text, so blanks and "N/A" stay different
txns_text = pd.read_csv("agent_transactions.csv", dtype=str, keep_default_na=False)

print("Transactions:", txns_raw.shape, "| Agents:", agents.shape)
print()
txns_raw.info()

Transactions: (612, 5) | Agents: (20, 3)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 612 entries, 0 to 611
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   txn_id    612 non-null    object 
 1   agent_id  612 non-null    object 
 2   txn_type  612 non-null    object 
 3   amount    590 non-null    float64
 4   txn_date  612 non-null    object 
dtypes: float64(1), object(4)
memory usage: 24.0+ KB


In [3]:
# -- A first look at the text columns --
print(txns_raw["txn_type"].value_counts())
print()
print("Agents in the register:", agents["agent_id"].nunique(), "| unique ids:", agents["agent_id"].is_unique)
print("Agent ids in the transactions:", txns_raw["agent_id"].nunique())
print("Dates look like:", txns_text["txn_date"].head(3).tolist())
txns_raw.head()

txn_type
deposit       142
withdrawal    131
Deposit       128
DEPOSIT       111
Withdrawal    100
Name: count, dtype: int64

Agents in the register: 20 | unique ids: True
Agent ids in the transactions: 21
Dates look like: ['2026-03-03', '2026-04-27', '2026-04-29']


,txn_id,agent_id,txn_type,amount,txn_date
0,TX00001,AG009,deposit,353.0,2026-03-03
1,TX00002,AG020,deposit,1238.0,2026-04-27
2,TX00003,AG008,DEPOSIT,9129.0,2026-04-29
3,TX00004,AG009,deposit,652.0,2026-04-24
4,TX00005,AG009,withdrawal,478.0,2026-04-17


## 2. The data-quality audit

Each row below is one problem, the dimension it breaks, the column it sits in and the number of rows affected. Every count is worked out with code on the **raw** file (so the 12 repeated rows are still included).

In [4]:
# -- The audit table: every count comes from the raw data --
amount_numbers = pd.to_numeric(txns_raw["amount"], errors="coerce").dropna()
q1, q3 = amount_numbers.quantile([0.25, 0.75])
upper_fence = q3 + 1.5 * (q3 - q1)
all_dates = pd.to_datetime(txns_text["txn_date"].str.replace("/", "-", regex=False))

audit = pd.DataFrame([
    ("Repeated transactions (same txn_id appears more than once)", "Uniqueness", "txn_id",
     txns_raw.duplicated(subset="txn_id").sum()),
    ("Transaction type not written as Deposit or Withdrawal", "Consistency", "txn_type",
     (~txns_raw["txn_type"].isin(["Deposit", "Withdrawal"])).sum()),
    ("Amount left blank", "Completeness", "amount",
     (txns_text["amount"] == "").sum()),
    ("Amount typed as the text N/A", "Validity", "amount",
     (txns_text["amount"] == "N/A").sum()),
    ("Amount of exactly 250,000 (identical, round, far above the rest)", "Accuracy", "amount",
     (txns_text["amount"] == "250000").sum()),
    ("Amount above the IQR upper fence", "Validity", "amount",
     (amount_numbers > upper_fence).sum()),
    ("Date written with / instead of -", "Consistency", "txn_date",
     txns_text["txn_date"].str.contains("/", regex=False).sum()),
    ("agent_id missing from the agent register", "Validity", "agent_id",
     (~txns_raw["agent_id"].isin(agents["agent_id"])).sum()),
    ("Date outside 1 March to 30 April 2026", "Timeliness", "txn_date",
     ((all_dates < "2026-03-01") | (all_dates > "2026-04-30")).sum()),
], columns=["issue", "dimension", "column", "rows_affected"])

audit["rows_affected"] = audit["rows_affected"].astype(int)
audit

,issue,dimension,column,rows_affected
0,Repeated transactions (same txn_id appears more than once),Uniqueness,txn_id,12
1,Transaction type not written as Deposit or Withdrawal,Consistency,txn_type,384
2,Amount left blank,Completeness,amount,15
3,Amount typed as the text N/A,Validity,amount,7
4,"Amount of exactly 250,000 (identical, round, far above the rest)",Accuracy,amount,5
5,Amount above the IQR upper fence,Validity,amount,40
6,Date written with / instead of -,Consistency,txn_date,5
7,agent_id missing from the agent register,Validity,agent_id,8
8,Date outside 1 March to 30 April 2026,Timeliness,txn_date,0


In [5]:
# -- A detail worth knowing: repeated ids are not always identical rows --
print("Exact duplicate rows:     ", txns_raw.duplicated().sum())
print("Repeated transaction ids: ", txns_raw.duplicated(subset="txn_id").sum())

Exact duplicate rows:      11
Repeated transaction ids:  12


**What the audit shows.** All six dimensions were checked. The two numbers above also reveal a trap: there are 12 repeated transaction ids but only 11 exact duplicate rows. One repeated transaction had its `agent_id` changed in one of its copies, so `drop_duplicates()` with no arguments would have left a duplicate behind. The cleaning function therefore defines a duplicate by the key, `txn_id`. The timeliness check found no dates outside the expected 60-day window.

## 3. Clean the data with the pipeline function

All the cleaning rules live in `cleaning_pipeline.py` in one function, `clean_agent_data(txns, agents)`. It works on a copy, uses no row numbers, and returns the clean table plus a log. Calling it here keeps the notebook short and means next month's extract can be cleaned with the same one line.

In [6]:
# -- Run the pipeline and read the log --
clean_df, log = clean_agent_data(txns_raw, agents)

for line in log:
    print("-", line)

print()
print("Rows before cleaning:", len(txns_raw), "| rows after cleaning:", len(clean_df))

- Start: 612 transaction rows
- Removed 12 duplicate transactions (same txn_id)
- Standardised txn_type on 376 rows; 0 rows had an unknown type
- Converted 5 dates written with '/' to ISO format; 0 dates could not be read
- Flagged and filled 20 missing or invalid amounts with the median (KES 1,776.50)
- Flagged 40 outliers outside KES 0 to 6,084 (flagged, not removed)
- Merged with the agent register: 600 rows before, 600 after; 7 transactions have an unknown agent (kept, county = 'Unknown')
- Finish: 600 clean rows

Rows before cleaning: 612 | rows after cleaning: 600


In [7]:
# -- Check the cleaned columns --
print(clean_df["txn_type"].value_counts())
print()
print("Unique txn_id:", clean_df["txn_id"].is_unique)
print()
clean_df.dtypes

txn_type
Deposit       373
Withdrawal    227
Name: count, dtype: int64

Unique txn_id: True



,0
txn_id,object
agent_id,object
txn_type,object
amount,float64
txn_date,datetime64[ns]
amount_imputed,bool
is_outlier,bool
county,object
agent_known,bool


**Uniqueness.** 12 repeated transactions were removed (612 rows became 600). I kept the first copy of each, because a transaction id is a unique key and a second copy is a system repeat, not a second payment.

**Consistency.** The five spellings of the transaction type (such as `DEPOSIT ` with a trailing space, `withdrawal` and `deposit`) were reduced to exactly two values, `Deposit` and `Withdrawal`. 376 rows needed a change, and no row was left with an unknown type. Five dates written with `/` were converted to the same ISO format as the rest.

**Validity and types.** `amount` is now numeric and `txn_date` is a real date. Because pandas reads `N/A` as missing, both the blanks and the `N/A` values arrive as `NaN`, and the pipeline treats them in one step.

## 4. Completeness: missing and invalid amounts

**Decision.** 20 amounts (15 blank and 5 `N/A`, which is 3.3% of the 600 transactions) have no usable value. I did **not** delete these rows, because each one is a real transaction and removing them would hide it from the counts. I **flagged** them first in `amount_imputed` and then filled them with the **median** of the real amounts (KES 1,776.50). The median is the right choice because amounts are strongly skewed (a few large values pull the mean up), so the median is a more typical value than the mean. The flag means anyone can exclude the estimates later.

**Effect.** The filled values add KES 35,530 to the totals, which is 2.5% of the value in the summary tables, so they cannot change the ranking of the counties by much, but they are estimates, not facts.

## 5. Outliers: the IQR method and the 250,000 values

In [8]:
# -- IQR fences worked out from the real (not filled) amounts --
observed = clean_df.loc[~clean_df["amount_imputed"], "amount"]
q1, q3 = observed.quantile([0.25, 0.75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr

print(f"Q1 = {q1:,.0f}, Q3 = {q3:,.0f}, IQR = {iqr:,.0f}, upper fence = {upper:,.0f}")
print("Transactions flagged as outliers:", clean_df["is_outlier"].sum())
print()
print("The biggest amounts:")
print(clean_df.sort_values("amount", ascending=False)
      [["txn_id", "agent_id", "txn_type", "amount", "txn_date"]].head(8).to_string(index=False))

Q1 = 1,077, Q3 = 3,080, IQR = 2,003, upper fence = 6,084
Transactions flagged as outliers: 40

The biggest amounts:
 txn_id agent_id   txn_type   amount   txn_date
TX00595    AG016    Deposit 250000.0 2026-03-11
TX00316    AG014    Deposit 250000.0 2026-04-01
TX00425    AG005 Withdrawal 250000.0 2026-03-02
TX00464    AG009 Withdrawal 250000.0 2026-04-11
TX00307    AG012    Deposit 250000.0 2026-03-02
TX00184    AG006    Deposit  28847.0 2026-03-08
TX00223    AG011    Deposit  22316.0 2026-03-02
TX00574    AG017    Deposit  16605.0 2026-03-16


In [9]:
# -- Investigating the 250,000 values --
suspect = clean_df["amount"] == 250000

print("Transactions of exactly KES 250,000:", suspect.sum())
print("Median amount:", observed.median())
print("Share of all cleaned value:", round(clean_df.loc[suspect, "amount"].sum() / clean_df["amount"].sum() * 100, 1), "%")
print("Average amount with them:   ", round(clean_df["amount"].mean()))
print("Average amount without them:", round(clean_df.loc[~suspect, "amount"].mean()))
print()
print(clean_df[suspect][["txn_id", "agent_id", "txn_type", "txn_date"]].to_string(index=False))

Transactions of exactly KES 250,000: 5
Median amount: 1776.5
Share of all cleaned value: 46.5 %
Average amount with them:    4481
Average amount without them: 2418

 txn_id agent_id   txn_type   txn_date
TX00307    AG012    Deposit 2026-03-02
TX00316    AG014    Deposit 2026-04-01
TX00425    AG005 Withdrawal 2026-03-02
TX00464    AG009 Withdrawal 2026-04-11
TX00595    AG016    Deposit 2026-03-11


**Finding.** The IQR rule flagged 40 transactions. Most are large but believable amounts (a skewed spread of money amounts always has a long tail). The 5 transactions of **exactly KES 250,000** are different: they are identical, perfectly round and about 141 times the median, and together they make up 46.5% of all the value in the file. That looks like a placeholder or a system limit, not 5 real customers who happened to move the same amount.

**Recommendation.** (1) Do not delete them, because a real transaction may be hiding behind each one. (2) Keep them in the clean file with `is_outlier = True`. (3) **Quarantine** them from the summary tables until the data owner confirms what 250,000 means in the legacy system. (4) Ask the legacy-system team whether it is a default or a cap, and correct the values from the source records. The summaries below follow this rule: they leave out those 5 rows and the impact is shown separately.

## 6. Merge with the agent register and validate it

The pipeline joins with a **left join** so that no transaction is lost, and it uses `validate="many_to_one"` so pandas stops with an error if the register ever contains a repeated agent id.

In [10]:
# -- Post-merge validation --
rows_before = txns_raw["txn_id"].nunique()        # one row per distinct transaction
rows_after = len(clean_df)
unknown = clean_df[~clean_df["agent_known"]]

print("Distinct transactions before the merge:", rows_before)
print("Rows after the merge:", rows_after, "| same count:", rows_before == rows_after)
print("Transactions with an unknown agent:", len(unknown), "| agent id(s):", unknown["agent_id"].unique().tolist())
print("Value of those transactions: KES", f"{unknown['amount'].sum():,.0f}")
print()
print("Rows an inner join would have lost:", len(unknown), "of", rows_after)
print()
print(clean_df["county"].value_counts())

Distinct transactions before the merge: 600
Rows after the merge: 600 | same count: True
Transactions with an unknown agent: 7 | agent id(s): ['AG099']
Value of those transactions: KES 27,156

Rows an inner join would have lost: 7 of 600

county
Nairobi    180
Mombasa    133
Kisumu     111
Nakuru      99
Kiambu      70
Unknown      7
Name: count, dtype: int64


**Result.** The merge kept all 600 rows. 7 transactions (KES 27,156) belong to agent `AG099`, who is not in the agent register (8 rows in the raw file, one of which was a repeated copy). Dropping them with an inner join would have silently removed that value, so I **kept them** with the county **Unknown** and a flag column, `agent_known`. The governance team should check whether `AG099` is a new agent who has not been registered yet or a typing mistake.

## 7. Summary tables

These tables use the trusted rows: everything except the five suspected 250,000 values. The imputed amounts are included and flagged.

In [11]:
# -- (a) County summary, split by transaction type --
trusted = clean_df[clean_df["amount"] != 250000].copy()

county_summary = (trusted.groupby(["county", "txn_type"])
                  .agg(total_value=("amount", "sum"),
                       transactions=("txn_id", "count"),
                       avg_value=("amount", "mean"))
                  .round(0)
                  .reset_index())
county_summary

,county,txn_type,total_value,transactions,avg_value
0,Kiambu,Deposit,84737.0,42,2018.0
1,Kiambu,Withdrawal,77050.0,27,2854.0
2,Kisumu,Deposit,186420.0,73,2554.0
3,Kisumu,Withdrawal,76374.0,37,2064.0
4,Mombasa,Deposit,148986.0,72,2069.0
5,Mombasa,Withdrawal,118592.0,60,1977.0
6,Nairobi,Deposit,293590.0,117,2509.0
7,Nairobi,Withdrawal,162036.0,62,2613.0
8,Nakuru,Deposit,184890.0,60,3082.0
9,Nakuru,Withdrawal,78594.0,38,2068.0


In [12]:
# -- County totals, ranked --
county_totals = (trusted.groupby("county")
                 .agg(total_value=("amount", "sum"),
                      transactions=("txn_id", "count"),
                      avg_value=("amount", "mean"))
                 .round(0)
                 .sort_values("total_value", ascending=False))
county_totals["share_pct"] = (county_totals["total_value"] / county_totals["total_value"].sum() * 100).round(1)
county_totals

,total_value,transactions,avg_value,share_pct
county,,,,
Nairobi,455626.0,179,2545.0,31.7
Mombasa,267578.0,132,2027.0,18.6
Nakuru,263484.0,98,2689.0,18.3
Kisumu,262795.0,110,2389.0,18.3
Kiambu,161788.0,69,2345.0,11.2
Unknown,27156.0,7,3879.0,1.9


In [13]:
# -- (b) Weekly transaction value by county --
# week_start is the Monday of each week
trusted["week_start"] = (trusted["txn_date"]
                         - pd.to_timedelta(trusted["txn_date"].dt.dayofweek, unit="D")).dt.date

weekly_pivot = trusted.pivot_table(index="week_start", columns="county",
                                   values="amount", aggfunc="sum", fill_value=0).round(0)
weekly_pivot

county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru,Unknown
week_start,,,,,,
2026-02-23,0.0,1893.0,6759.0,7046.0,2744.0,0.0
2026-03-02,18317.0,37933.0,35670.0,68967.0,56574.0,0.0
2026-03-09,17772.0,23186.0,30478.0,52860.0,28730.0,0.0
2026-03-16,22054.0,28125.0,25144.0,43108.0,39795.0,1639.0
2026-03-23,11504.0,27077.0,17050.0,41374.0,18998.0,0.0
2026-03-30,22609.0,41077.0,38344.0,32931.0,18145.0,8507.0
2026-04-06,15030.0,15341.0,47092.0,56786.0,26609.0,9824.0
2026-04-13,12265.0,29416.0,26730.0,56432.0,26152.0,0.0
2026-04-20,24507.0,37439.0,16687.0,65187.0,30862.0,1877.0


In [14]:
# -- What the quarantine and the estimates do to the totals --
print("Total value, all cleaned rows:                KES", f"{clean_df['amount'].sum():,.0f}")
print("Total value, without the 250,000 values:      KES", f"{trusted['amount'].sum():,.0f}")
print("Total value, real amounts only (no estimates): KES", f"{trusted.loc[~trusted['amount_imputed'], 'amount'].sum():,.0f}")

Total value, all cleaned rows:                KES 2,688,427
Total value, without the 250,000 values:      KES 1,438,427
Total value, real amounts only (no estimates): KES 1,402,897


**What the tables say.** Nairobi handles the most value (KES 455,626 from 179 transactions, 31.7% of the total), followed by Mombasa (KES 267,578). Kiambu is the smallest of the registered counties at KES 161,788. Deposits are 64.2% of the value (370 deposits worth KES 923,902 against 225 withdrawals worth KES 514,524). The best single agent is `AG006` with KES 110,018. The first and last weeks in the pivot table are partial weeks (the extract starts on a Sunday and ends on a Wednesday), so they should not be compared with the full weeks.

## 8. Save the clean data

In [15]:
# -- Save the clean transactions --
clean_df.to_csv("outputs/agent_transactions_clean.csv", index=False, date_format="%Y-%m-%d")

# read it back and check the promises in the brief
check = pd.read_csv("outputs/agent_transactions_clean.csv")
print("Rows:", len(check))
print("Unique txn_id:", check["txn_id"].is_unique)
print("txn_type values:", sorted(check["txn_type"].unique()))
print("Date example:", check["txn_date"].iloc[0], "| amount type:", check["amount"].dtype)
print("Columns:", check.columns.tolist())

Rows: 600
Unique txn_id: True
txn_type values: ['Deposit', 'Withdrawal']
Date example: 2026-03-03 | amount type: float64
Columns: ['txn_id', 'agent_id', 'txn_type', 'amount', 'txn_date', 'amount_imputed', 'is_outlier', 'county', 'agent_known']


# Data Quality Report

**Scope.** The agent transaction extract (612 raw rows, 1 March to 29 April 2026) and the register of 20 agents. After cleaning there are 600 rows. Every rule is applied by `clean_agent_data()`, and the log above records each step.

| Dimension | Finding | Action | Residual risk |
|---|---|---|---|
| **Completeness** | 20 of 600 transactions (3.3%) had no usable amount: 15 blank and 5 `N/A`. | Flagged in `amount_imputed`, filled with the median (KES 1,776.50). | The estimates add KES 35,530 of value that is not real; exclude the flag if exact totals are needed. |
| **Validity** | `N/A` typed in the amount column; agent `AG099` is not in the register (7 transactions, KES 27,156). | Converted to numbers; unknown agents kept with county `Unknown` and `agent_known = False`. | The register may be out of date, or `AG099` may be a typing mistake. |
| **Accuracy** | 5 transactions of exactly KES 250,000 (46.5% of all value) look like a placeholder or system limit. 40 transactions are IQR outliers in total. | Flagged with `is_outlier`, kept in the file, left out of the summary tables. | Not confirmed with the data owner, so the true value of these rows is unknown. |
| **Consistency** | Five spellings of the transaction type (capitals and trailing spaces); 5 dates written with `/`. | Standardised to `Deposit` and `Withdrawal` and ISO dates (`YYYY-MM-DD`). | Low: no unknown types or unreadable dates remain. |
| **Uniqueness** | 12 repeated `txn_id` values; only 11 were exact duplicates because one copy had a different `agent_id`. | Removed by `txn_id`, keeping the first copy. | Low, but the legacy system should be asked why it repeats rows. |
| **Timeliness** | All dates fall between 1 March and 29 April 2026, with none in the future. | None needed. | The first and last weeks are partial, and 60 days is a short period for judging trends. |

**Privacy.** This extract has no customer details, but if it also held customer phone numbers, then the **phone number**, the **customer name** and any **national ID number** would be personal data under the Kenya Data Protection Act 2019, and a transaction history tied to a phone number reveals a person's financial behaviour. I would apply data minimisation first (not load the phone number at all unless the analysis needs it), then replace identifiers with a salted hash (pseudonymisation) so customers can still be counted without being named, mask any number shown on screen (for example `0712****78`), and report only aggregates, hiding any group with fewer than about five people. The salt and the raw files would stay out of the repository (they are in `.gitignore`), access would be limited to the people who need it, and the data would be deleted once the purpose is complete. Agent identifiers also need care, because an agent is often a named sole trader.